# dfthcsos (chemistry): block encoding

Builds and checks only this task, writing it to the `submission/` folder.
The same code runs inside the merged notebook `vqcc_submission.ipynb`,
which builds all three tasks and writes `submission.zip`.

In [ ]:
# ---- Settings ---------------------------------------------------------------
VERIFY = 'quick'      # 'none' | 'quick' (minutes) | 'full' (official verifiers, ~30-60 min)
USE_IMPROVED = {'invsqrt': True, 'dqi': True, 'dfthcsos': False}
REQUIRE_FRONTIER = True   # stop with an error if the frontier circuits can't be built (instead of a much lower score)

### Optional: run locally

On Kaggle, leave this as `None`: the data is found with `kagglehub`.
To run on your own machine, set `LOCAL_DATA` to the folder where you unpacked the
competition data (the folder that contains `verifiers/` and `sample_submission/`,
or any folder above it).

In [ ]:
LOCAL_DATA = None     # e.g. 'data/'

## 1. Setup

Finds the competition data, imports kickmix and the official verifiers, and
prepares the output folder. Data is looked up in this order:

1. `LOCAL_DATA` (the optional cell above), or the `VQCC_DATA` environment variable;
2. `kagglehub.competition_download('google-verifiable-quantum-circuit-compilation')`;
3. any copy mounted under `/kaggle/input`.

kickmix is imported and checked: if the import picks up the data's `kickmix/` folder
instead of the real module (the cause of the "no attribute CircuitBuilder" errors),
the compiled `kickmix*.so` from the data is loaded directly.

Two public datasets are downloaded with `kagglehub.dataset_download`:
`ahoirg/qcc-reproducible-circuit-assets` (frontier circuits, section 1b) and
`marculera/dfthc-compact-pipeline` (chemistry factorization pipeline).

In [ ]:
import glob
import io
import itertools
import json
import math
import os
import shutil
import sys
import time
import zipfile
from dataclasses import dataclass
from fractions import Fraction
from pathlib import Path

import numpy as np


COMPETITION = 'google-verifiable-quantum-circuit-compilation'
MARKER = Path('verifiers') / 'invsqrt' / 'verify_submission_invsqrt.py'


def _data_root_under(folder) -> Path | None:
    """The folder holding verifiers/ (the given folder or one below it), if any."""
    folder = Path(folder).expanduser().resolve()
    if (folder / MARKER).exists():
        return folder
    for hit in sorted(folder.glob('**/' + MARKER.as_posix())):
        return hit.parents[2]
    return None


def find_data_root() -> Path:
    tried = []
    for folder in (LOCAL_DATA, os.environ.get('VQCC_DATA')):
        if folder:
            tried.append(str(folder))
            root = _data_root_under(folder) if Path(folder).expanduser().exists() else None
            if root:
                return root
    try:
        import kagglehub
        path = kagglehub.competition_download(COMPETITION)
        print('kagglehub:', path)
        tried.append(f'kagglehub ({path})')
        root = _data_root_under(path)
        if root:
            return root
    except Exception as ex:
        tried.append(f'kagglehub failed: {ex}')
    if Path('/kaggle/input').exists():
        tried.append('/kaggle/input')
        root = _data_root_under('/kaggle/input')
        if root:
            return root
    raise FileNotFoundError('Competition data not found. Tried: ' + '; '.join(tried) +
                            '. On Kaggle, attach the competition data; locally, set LOCAL_DATA.')


KICKMIX_API = ('CircuitBuilder', 'Circuit', 'GF2Field')


def _is_kickmix(module):
    return all(hasattr(module, name) for name in KICKMIX_API)


def _find_files(root, pattern, max_depth=6):
    """Files under root matching pattern, following symlinked folders (Path.glob('**') does not)."""
    import fnmatch
    root, hits = Path(root), []
    if not root.is_dir():
        return hits
    base = len(root.parts)
    for folder, dirs, files in os.walk(root, followlinks=True):
        if len(Path(folder).parts) - base >= max_depth:
            dirs[:] = []
        hits += [Path(folder) / f for f in files if fnmatch.fnmatch(f, pattern)]
    return sorted(hits)


def load_kickmix(data_root):
    """Return the real kickmix module.

    A plain `import kickmix` can pick up a *folder* named kickmix (such as the one in
    the competition data) as an empty namespace package, which then has no
    CircuitBuilder. So the imported module is checked, and if it is not the real one,
    the compiled kickmix*.so shipped with the data is loaded from its file path.
    """
    import importlib
    import importlib.util
    tried = []
    try:
        import kickmix
        if _is_kickmix(kickmix):
            return kickmix
        where = getattr(kickmix, '__file__', None) or list(getattr(kickmix, '__path__', []))
        tried.append(f'import kickmix gave {where} without {KICKMIX_API}')
    except ImportError as ex:
        tried.append(f'import kickmix: {ex}')
    sys.modules.pop('kickmix', None)
    roots = [Path(data_root)] + ([Path('/kaggle/input')] if Path('/kaggle/input').exists() else [])
    for so in [p for r in roots for p in _find_files(r, 'kickmix*.so')]:
        try:
            spec = importlib.util.spec_from_file_location('kickmix', so)
            module = importlib.util.module_from_spec(spec)
            sys.modules['kickmix'] = module
            spec.loader.exec_module(module)
            if _is_kickmix(module):
                return module
            tried.append(f'{so}: loaded but lacks {KICKMIX_API}')
        except Exception as ex:
            tried.append(f'{so}: {type(ex).__name__}: {ex}')
        sys.modules.pop('kickmix', None)
    for wheel in [p for r in roots for p in _find_files(r, 'kickmix*.whl')]:
        import subprocess
        res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', str(wheel)],
                             capture_output=True, text=True)
        importlib.invalidate_caches()
        try:
            import kickmix
            if _is_kickmix(kickmix):
                return kickmix
        except ImportError:
            pass
        sys.modules.pop('kickmix', None)
        tried.append(f'{wheel}: pip install -> {res.returncode} {res.stderr.strip()[-200:]}')
    folder = Path(data_root) / 'kickmix'
    listing = sorted(p.name for p in folder.iterdir()) if folder.is_dir() else 'missing'
    raise ImportError(f'kickmix not usable (Python {sys.version.split()[0]}). Tried: ' + '; '.join(tried) +
                      f'. Contents of {folder}: {listing}')


def download_dataset(handle):
    """kagglehub.dataset_download(handle), or None (with the reason printed)."""
    try:
        import kagglehub
        path = kagglehub.dataset_download(handle)
        print(f'{handle}: {path}')
        return Path(path)
    except Exception as ex:
        print(f'{handle}: kagglehub download failed ({type(ex).__name__}: {ex})')
        return None


DATA = find_data_root()
VERIFIERS = DATA / 'verifiers'
SAMPLE = DATA / 'sample_submission'
km = load_kickmix(DATA)
sys.path.insert(0, str(VERIFIERS))

# Public datasets: frontier circuit assets (section 1b) and the chemistry pipeline.
ASSETS_PATH = download_dataset('ahoirg/qcc-reproducible-circuit-assets')
CHEM_PIPELINE_PATH = download_dataset('marculera/dfthc-compact-pipeline')

OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('output').resolve()
SUB = OUT / 'submission'
if SUB.exists():
    shutil.rmtree(SUB)
(SUB / 'dfthcsos').mkdir(parents=True)
REPORT = {}
print('data:', DATA)
print('kickmix:', km.__file__, '| version', getattr(km, '__version__', '?'), '| Python', sys.version.split()[0])
print('output:', SUB)

## 1b. Public frontier components (credited)

The strongest public circuits for all three tasks are built by many Kaggle
participants and shared under Apache-2.0. We start from them, verify every one
with the official verifiers, and keep our own circuits (sections 2-4) as the
fallback. They come from **ahoirg's [QCC Reproducible Circuit Assets](https://www.kaggle.com/datasets/ahoirg/qcc-reproducible-circuit-assets), release 2**
(downloaded with `kagglehub.dataset_download` in the setup cell):

| Task | Component | Credits (see the dataset's CREDITS files for the full lineage) |
|---|---|---|
| invsqrt | 62-qubit circuit with five proved rewrites | mrmeows / team selenium (architecture), busyaprime, yoyobar, Ahmed Berat Özer, Ehan Mallick, SomeJoint; rewrite matcher by Kirill Tymchishin; lineage heliosli, Phuc Thai, Dmitrii Gluzdov, marsyas |
| dqi | 891-qubit decoder | SomeJoint / Agniv Banerjee and the team credited in their source (Ahmed Berat Özer, Alex Chilton, jday96314, mrmeows, manwithacat, Dmitrii Gluzdov, ViberTeam, Phuc Thai), yoyobar, ahoirg |
| dfthcsos | C39 factorization + paired-product loaders | marculera (factorization pipeline, tensorqrom), Syed Asad Ali / SomeJoint (C39 loaders), mrmeows, heliosli, Ahmed Berat Özer, kozykappa, Michael Rustad and others |

What this notebook runs from that dataset (all hash-checked against the
dataset's manifest first):

- `inverse_factor.py`: an exact five-window gate rewrite of the invsqrt circuit,
  each edit proved on all 128 inputs of its 7 wires.
- `scheduled_emit.py` and `tensorqrom.py`: emit the 15 chemistry loaders from the
  C39 angle table and its algebraic plans.
- `v2-dqi.kmb.xz`: the prebuilt dqi decoder, decompressed.

In [ ]:
USE_FRONTIER = {'invsqrt': True, 'dqi': True, 'dfthcsos': True}
LOCAL_ASSETS = None          # local folder holding qcc-reproducible-circuit-assets(-v2); found automatically on Kaggle
ASSETS_DATASET = 'ahoirg/qcc-reproducible-circuit-assets'


def _sha256(path):
    import hashlib
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 24), b''):
            h.update(chunk)
    return h.hexdigest()


def find_assets():
    """Folder holding release 2 of the assets (v2-assets-manifest.json), or None."""
    roots = [Path(p) for p in (LOCAL_ASSETS, os.environ.get('VQCC_ASSETS'), ASSETS_PATH) if p]
    roots.append(Path('/kaggle/input'))
    for root in roots:
        if root.exists():
            for hit in sorted(root.expanduser().glob('**/v2-assets-manifest.json')):
                return hit.parent
    return None


def load_frontier():
    """Authenticate the assets, extract the source bundle and build the frontier
    invsqrt and chemistry circuits. Returns a dict of paths, or raises."""
    assets = find_assets()
    if assets is None:
        raise FileNotFoundError(f'attach the {ASSETS_DATASET} dataset (release 2)')
    manifest = json.loads((assets / 'v2-assets-manifest.json').read_text())
    for name, pin in manifest['files'].items():
        if (assets / name).exists() and _sha256(assets / name) != pin['sha256']:
            raise ValueError(f'{name}: SHA-256 differs from the assets manifest')
    work = OUT / 'frontier_build'
    if work.exists():
        shutil.rmtree(work)
    src = work / 'source'
    src.mkdir(parents=True)
    import hashlib
    bundle = json.loads((assets / 'v2-source-bundle.json').read_text())
    for name, entry in bundle['files'].items():
        if hashlib.sha256(entry['text'].encode()).hexdigest() != entry['sha256']:
            raise ValueError(f'source bundle member {name} fails its hash')
        (src / name).parent.mkdir(parents=True, exist_ok=True)
        (src / name).write_text(entry['text'])
    sys.path.insert(0, str(src))
    from inverse_factor import factor_windows, prove_edits
    from scheduled_emit import build_circuit
    pins = json.loads((src / 'manifest.json').read_text())
    # invsqrt: base circuit -> five proved rewrites
    base = assets / 'v2-inverse-control.kmb'
    text = work / 'inverse_control.kmx'
    km.Circuit.from_file(str(base), format='kmb').to_file(str(text), format='kmx')
    lines, edits = factor_windows(text.read_text().splitlines())
    proof = prove_edits(edits)
    inv_path = work / 'invsqrt.kmb'
    tmp = work / 'inverse_transformed.kmx'
    tmp.write_text('\n'.join(lines) + '\n')
    km.Circuit.from_file(str(tmp), format='kmx').to_file(str(inv_path), format='kmb')
    # chemistry: 15 loaders from the C39 table and its plans
    chem = work / 'dfthcsos'
    chem.mkdir()
    factor = assets / 'v2-c39-factor.npz'
    with np.load(factor, allow_pickle=False) as z:
        words = z['angle_words']
    for slot in range(15):
        plan = json.loads((src / 'plans' / f'slot-{slot}.json').read_text())
        cur = words[:, slot * 5:slot * 5 + 5]
        prev = words[:, (slot - 1) * 5:slot * 5] if slot else np.zeros_like(cur)
        delta = np.vstack([cur ^ prev, np.asarray(plan['unused_address_completion'], dtype=np.int64)])
        terms = [tuple(int(v, 16) for v in t) for t in plan['terms']]
        build_circuit(delta, terms, plan['low_cache_schedule']).to_file(str(chem / f'angle_k_{slot}.kmb'), format='kmb')
    shutil.copyfile(factor, chem / 'submission.npz')
    pinned = {**{k: v['sha256'] for k, v in pins['output_members'].items()}}
    mism = [k for k, h in pinned.items() if (work / k).exists() and _sha256(work / k) != h]
    if mism:
        raise ValueError(f'rebuilt files differ from the pinned outputs: {mism}')
    # dqi: decompress the prebuilt decoder
    import lzma
    dqi_path = work / 'dqi.kmb'
    with lzma.open(assets / 'v2-dqi.kmb.xz') as fin, open(dqi_path, 'wb') as fout:
        shutil.copyfileobj(fin, fout, 1 << 24)
    if _sha256(dqi_path) != manifest['build_inputs']['dqi']['sha256']:
        raise ValueError('decompressed dqi.kmb fails its hash')
    print('frontier built:', dict(proof=proof, files_checked=len(pinned) + 1))
    return dict(invsqrt=inv_path, dqi=dqi_path, dfthcsos=chem)


FRONTIER = {}
if any(USE_FRONTIER.values()):
    try:
        FRONTIER = load_frontier()
    except Exception as ex:
        if REQUIRE_FRONTIER:
            raise RuntimeError('frontier circuits could not be built, so the submission would score far '
                               'lower. Fix the cause below, or set REQUIRE_FRONTIER = False to continue '
                               f'with our own circuits. Cause: {type(ex).__name__}: {ex}') from ex
        print('frontier components unavailable, using our own circuits:', ex)

## 4. dfthcsos: chemistry block encoding

**Contract.** Submit a factorization of the FeMoCo Hamiltonian (`submission.npz`)
plus the circuits `angle_k_0 ... angle_k_{K-1}` that load each vector's Givens
angles as `brot`-bit words, `J` words per circuit. The verifier simulates the
circuits, rebuilds the Hamiltonian from the words they load, and requires both
residuals to be at most 20 mHa.
Cost = lambda_eff x Toffolis x qubits for the whole block encoding.

**Baseline:** brot = 12, J = 8, K = 10 circuits, each a plain lookup over 3,976
addresses. The angle circuits are 85% of the Toffolis, so fewer bits per angle
means more angles per circuit and fewer circuits:

| brot | J (128-qubit cap) | K | angle Toffolis |
|---|---|---|---|
| 12 | 8 | 10 | 79,480 |
| 11 | 9 | 9 | 71,532 |
| 10 | 10 | 8 | 63,584 |

Lower precision raises the two-body error (11 bits: 25.0 mHa with plain rounding,
over the limit). Our fix is to round the two-body vectors first, freeze them, then
refit the continuous weights to those rounded vectors with the starter kit's
trainer, so the weights absorb the rounding error (optional cell below; needs JAX).

The pipeline below works for any factorization and precision. It rebuilds the
one-body rows as eigenvectors after the two-body rounding (as the starter kit
does), picks the largest J that fits the qubit cap, builds the circuits with
the competition's reference lookup builder, verifies them, and falls back to
the baseline if anything fails.

In [ ]:
DFT_BROT = 12                  # bits per angle word
DFT_FACTORIZATION = None       # None = the baseline in sample_submission/; or a path to an .npz
DFT_REFIT_EPOCHS = 0           # > 0: refit the weights to the rounded vectors first (needs JAX + starter kit)
DFT_REFIT_LR = 1e-5
STARTER_KIT = None             # local starter kit folder (the one holding starter_kit/); found automatically on Kaggle


def dft_log_depth_edges(n):
    """Breadth-first edges of the log-depth binary Givens tree (arXiv:2605.30455)."""
    edges, level, depth = [], [0], 0
    while len(edges) < n - 1:
        nxt = []
        for label in level:
            nxt.append(label)
            right = label + (1 << depth)
            if right < n:
                edges.append((label, right))
                nxt.append(right)
        level, depth = nxt, depth + 1
    return np.asarray(edges, dtype=np.int64).reshape(-1, 2)


def dft_tree_angles(vectors, edges):
    """theta / pi in [0, 2) such that the tree maps e_0 to each (normalized) row."""
    u = np.array(vectors, dtype=np.float64, ndmin=2)
    u /= np.linalg.norm(u, axis=1, keepdims=True)
    theta = np.zeros((u.shape[0], u.shape[1] - 1))
    for t in range(u.shape[1] - 2, -1, -1):
        p, c = edges[t]
        theta[:, t] = np.arctan2(u[:, c], u[:, p])
        u[:, p] = np.hypot(u[:, p], u[:, c])
        u[:, c] = 0.0
    return np.mod(theta / np.pi, 2.0)


def dft_words(theta_over_pi, brot):
    return np.floor(np.mod(theta_over_pi, 2.0) * (1 << (brot - 1)) + 0.5).astype(np.int64) % (1 << brot)


def dft_encode(inputs, edges, brot):
    """(N + R*B, N - 1) angle words: two-body rows rounded, then one-body rows as the
    eigenvectors of h1_prime built from the decoded two-body vectors."""
    from dfthcsos.scoreinfra.dfthc_math import reconstruct_u_vectors, reconstruct_x, build_h1_prime
    U = np.asarray(inputs.params['U'], dtype=np.float64)
    R, B, N = U.shape
    two = dft_words(dft_tree_angles(U.reshape(-1, N), edges), brot)
    vec = reconstruct_u_vectors(two / (1 << (brot - 1)), edges)
    vec /= np.linalg.norm(vec, axis=1, keepdims=True)
    X = reconstruct_x(vec.reshape(R, B, N), inputs.w)
    X = np.tril(X) + np.swapaxes(np.tril(X, k=-1), -1, -2)
    h1 = inputs.ref['oei'] - 0.5 * np.einsum('prrq->pq', inputs.ref['tei'])
    hp = build_h1_prime(h1, inputs.S, inputs.beta_1, inputs.ref['electrons'], inputs.W, X)
    one = dft_words(dft_tree_angles(np.linalg.eigh(hp)[1].T, edges), brot)
    return np.vstack([one, two])


def dft_batch_size(num_vectors, brot, cap=128):
    """Largest J with selection + J*brot + (selection - 1) unary-iteration ancillas <= cap."""
    sel = (num_vectors - 1).bit_length()
    return (cap - sel - (sel - 1)) // brot


def find_starter_kit():
    tried = []
    if STARTER_KIT:
        tried.append(str(STARTER_KIT))
        hits = sorted(Path(STARTER_KIT).expanduser().glob('**/dfthc/df_thc_sga.py'))
        if hits:
            return hits[0].parents[1]
    try:
        import kagglehub
        p = kagglehub.dataset_download('sohier/verifiable-circuit-compilation-starter-kit')
        hits = sorted(Path(p).glob('**/dfthc/df_thc_sga.py'))
        if hits:
            return hits[0].parents[1]
        tried.append(f'kagglehub ({p})')
    except Exception as ex:
        tried.append(f'kagglehub failed: {ex}')
    hits = sorted(Path('/kaggle/input').glob('**/dfthc/df_thc_sga.py')) if Path('/kaggle/input').exists() else []
    if hits:
        return hits[0].parents[1]
    raise FileNotFoundError('starter kit not found: ' + '; '.join(tried))


def dft_refit(seed_npz, brot, epochs, lr):
    """Round the two-body vectors to brot bits, freeze them, refit the weights.
    Returns the path of the refitted factorization."""
    kit = find_starter_kit()
    sys.path.insert(0, str(kit))
    import jax.numpy as jnp
    from dfthc import df_thc_sga
    from dfthc.train_df_thc_sga import Initialize, Train
    from starter.task1_factorization.train_df_thc_qaggle import load_seed_params, frozen_symmetry_params, save_solution
    fcidump = VERIFIERS / 'dfthcsos' / 'FeMoCo76o113e.fcidump'
    ref = Initialize(str(fcidump), -22140.27335)
    params, _ = load_seed_params(str(seed_npz), orbitals=ref['orbitals'])
    U = np.asarray(params['U'], dtype=np.float64)
    R, B, N = U.shape
    edges = dft_log_depth_edges(N)
    from dfthcsos.scoreinfra.dfthc_math import reconstruct_u_vectors
    Uq = reconstruct_u_vectors(dft_words(dft_tree_angles(U.reshape(-1, N), edges), brot) / (1 << (brot - 1)), edges)
    Uq = (Uq / np.linalg.norm(Uq, axis=1, keepdims=True)).reshape(R, B, N)
    shifts = np.load(VERIFIERS / 'dfthcsos' / 'FeMoCo76o113e.fixed_shifts.npz')
    frozen = {'U': jnp.asarray(Uq, dtype=df_thc_sga.dtype),
              'Hsym_lower': jnp.asarray(shifts['Hsym_lower'], dtype=df_thc_sga.dtype),
              'H1_I': jnp.asarray(shifts['H1_I'], dtype=df_thc_sga.dtype)}
    sol = Train(generating_set='Qaggle', method='DF-THC-SOS', epochs=epochs, learning_rate=lr, rho=1e-4,
                params={**params, **frozen}, ref=ref, print_interval=max(1, epochs // 10), epsilon_dfthc=0.0,
                frozen_params=frozen, lambda_floor=49.0, sos_gap_floor=3.914898, learning_rate_final=lr / 5)
    out = OUT / f'refit_b{brot}_e{epochs}.npz'
    if out.exists():
        out.unlink()
    save_solution(str(out), sol['params_best'], sol['values'], ref=ref, metadata={'brot': brot, 'epochs': epochs})
    return out


def run_dfthcsos():
    from dfthcsos.verify import load_inputs, verify as verify_dfthcsos
    sys.path.insert(0, str(VERIFIERS / 'dfthcsos' / 'example_submission'))
    from angle_k_reference_circuit import build_angle_k_family
    t0 = time.time()
    src = Path(DFT_FACTORIZATION) if DFT_FACTORIZATION else SAMPLE / 'dfthcsos' / 'submission.npz'
    if DFT_REFIT_EPOCHS > 0:
        src = dft_refit(src, DFT_BROT, DFT_REFIT_EPOCHS, DFT_REFIT_LR)
    inputs = load_inputs(str(src))
    edges = dft_log_depth_edges(inputs.num_orbitals)
    words = dft_encode(inputs, edges, DFT_BROT)
    J = dft_batch_size(inputs.num_vectors, DFT_BROT)
    circuits = build_angle_k_family(words, DFT_BROT, J)
    out = SUB / 'dfthcsos'
    for f in out.glob('*'):
        f.unlink()
    for k, c in enumerate(circuits):
        c.to_file(str(out / f'angle_k_{k}.kmb'), format='kmb')
    with np.load(src) as z:
        np.savez(out / 'submission.npz', **{k: z[k] for k in ('U', 'W', 'H2_I', 'H1_I', 'Hsym_lower')},
                 tree_edges=edges, angle_words=words)
    r = verify_dfthcsos(str(out))          # raises if any gate fails
    m = r['metrics']
    return dict(brot=DFT_BROT, batch_size=J, circuits=len(circuits), score=r['score'], lambda_eff=r['lambda_eff'],
                toffolis=r['toffoli_count'], qubits=r['num_qubits'], error_1_mHa=round(1e3 * m['error_1'], 3),
                error_2_mHa=round(1e3 * m['error_2'], 3), seconds=round(time.time() - t0, 1))


def use_baseline_dfthcsos():
    out = SUB / 'dfthcsos'
    for f in out.glob('*'):
        f.unlink()
    for f in sorted((SAMPLE / 'dfthcsos').iterdir()):
        shutil.copy(f, out / f.name)


def run_frontier_dfthcsos():
    from dfthcsos.verify import verify as verify_dfthcsos
    t0 = time.time()
    out = SUB / 'dfthcsos'
    for f in out.glob('*'):
        f.unlink()
    for f in sorted(Path(FRONTIER['dfthcsos']).iterdir()):
        shutil.copy(f, out / f.name)
    r = verify_dfthcsos(str(out))          # raises if any gate fails
    m = r['metrics']
    return dict(brot=r['brot'], batch_size=r['batch_size'], circuits=r['num_circuits'], score=r['score'],
                lambda_eff=r['lambda_eff'], toffolis=r['toffoli_count'], qubits=r['num_qubits'],
                error_1_mHa=round(1e3 * m['error_1'], 4), error_2_mHa=round(1e3 * m['error_2'], 4),
                seconds=round(time.time() - t0, 1))


REPORT['dfthcsos'] = None
if USE_FRONTIER['dfthcsos'] and 'dfthcsos' in FRONTIER:
    try:
        REPORT['dfthcsos'] = dict(version='frontier', **run_frontier_dfthcsos())
    except Exception as ex:
        print('dfthcsos: frontier submission rejected:', ex)
if REPORT['dfthcsos'] is None and USE_IMPROVED['dfthcsos']:
    try:
        REPORT['dfthcsos'] = dict(version='ours', **run_dfthcsos())
    except Exception as ex:
        print('dfthcsos: our submission rejected:', ex)
if REPORT['dfthcsos'] is None:
    use_baseline_dfthcsos()
    REPORT['dfthcsos'] = dict(version='baseline')
    if VERIFY in ('quick', 'full'):
        from dfthcsos.verify import verify as verify_dfthcsos
        r = verify_dfthcsos(str(SUB / 'dfthcsos'))
        REPORT['dfthcsos'].update(verifier_status=r['status'], score=r['score'])
print(json.dumps(REPORT['dfthcsos'], indent=2, default=str))

## Summary

In [ ]:
print(json.dumps(REPORT, indent=2, default=str))
for f in sorted(SUB.rglob('*')):
    if f.is_file():
        print(f'{f.relative_to(SUB)}  {f.stat().st_size / 1e6:.1f} MB')